Chatbot "La IA en el mundo empresarial"

El proceso tiene cuatro pasos: cargar el PDF, separar el texto en fragmentos, buscar los fragmentos relacionados con la pregunta y pedir al modelo que redacte una respuesta basada en ellos.

Es un prototipo para probar y analizar. No se ha entrenado un modelo propio y no se utiliza para medir si los estudiantes aprenden más.

## 1. Instalar las librerías



In [1]:
%pip -q install -U pypdf sentence-transformers scikit-learn pandas openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


## 2. Subir el PDF

Sube el PDF de la asignatura que se va a usar como fuente.

In [2]:
from google.colab import files
from pathlib import Path

cargas = files.upload()
rutas_pdf = [nombre for nombre in cargas if nombre.lower().endswith(".pdf")]

if not rutas_pdf:
    raise ValueError("No se ha cargado ningún PDF. Ejecuta de nuevo esta celda y selecciona al menos un archivo PDF.")

print("Documentos cargados:")
for ruta in rutas_pdf:
    print(" -", ruta)

Saving Unidad 1 - La IA en el mundo empresarial_cgranados.pdf to Unidad 1 - La IA en el mundo empresarial_cgranados.pdf
Saving Unidad 2 - Data Governance.pdf to Unidad 2 - Data Governance.pdf
Saving Unidad 3 - Behavioral Economics y Diseño.pdf to Unidad 3 - Behavioral Economics y Diseño.pdf
Saving Unidad 4 - Gestión de proyectos.pdf to Unidad 4 - Gestión de proyectos.pdf
Saving Unidad 5 - Aspectos éticos y legales en la Inteligencia Artificial.pdf to Unidad 5 - Aspectos éticos y legales en la Inteligencia Artificial.pdf
Documentos cargados:
 - Unidad 1 - La IA en el mundo empresarial_cgranados.pdf
 - Unidad 2 - Data Governance.pdf
 - Unidad 3 - Behavioral Economics y Diseño.pdf
 - Unidad 4 - Gestión de proyectos.pdf
 - Unidad 5 - Aspectos éticos y legales en la Inteligencia Artificial.pdf


## 3. Leer el texto del PDF

Se guarda el nombre del archivo y el número de página. Esto permite revisar la fuente de una respuesta.

In [3]:
import re
from pathlib import Path
from pypdf import PdfReader

def normalizar_texto(texto):
    # Unifico espacios y elimino caracteres nulos que pueden aparecer en el PDF.
    texto = (texto or "").replace("\\x00", " ")
    return re.sub(r"\\s+", " ", texto).strip()

def extraer_paginas(rutas):
    paginas = []
    paginas_vacias = []

    for ruta in rutas:
        try:
            lector = PdfReader(ruta)
            if lector.is_encrypted:
                raise ValueError("El PDF está protegido con contraseña.")
        except Exception as error:
            raise ValueError(
                f"No se ha podido abrir el archivo '{Path(ruta).name}'. "
                "Comprueba que sea un PDF válido y que no esté protegido."
            ) from error

        for numero, pagina in enumerate(lector.pages, start=1):
            try:
                texto = normalizar_texto(pagina.extract_text())
            except Exception:
                texto = ""

            if texto:
                paginas.append({
                    "archivo": Path(ruta).name,
                    "pagina": numero,
                    "texto": texto
                })
            else:
                paginas_vacias.append((Path(ruta).name, numero))

    return paginas, paginas_vacias

paginas, paginas_vacias = extraer_paginas(rutas_pdf)
print(f"Páginas con texto extraído: {len(paginas)}")

if paginas_vacias:
    print(f"Aviso: no se ha podido extraer texto de {len(paginas_vacias)} página(s).")

if not paginas:
    raise ValueError(
        "No se ha extraído texto de los PDF. Comprueba que contienen texto seleccionable. "
        "Si son escaneos, será necesario aplicar OCR antes de continuar."
    )


Páginas con texto extraído: 360


## 4. Dividir el texto en partes

El texto se divide en fragmentos de tamaño más pequeño para una mejor búsqueda de la información. El solapamiento hace que una parte del final de un fragmento aparezca también al comienzo del siguiente.

In [4]:
def crear_fragmentos(paginas, tamano_palabras=220, solapamiento=40):
    # Los valores se expresan en palabras, no en caracteres.
    if tamano_palabras < 1:
        raise ValueError("El tamaño del fragmento debe ser positivo.")
    if solapamiento < 0 or solapamiento >= tamano_palabras:
        raise ValueError("El solapamiento debe ser no negativo y menor que el tamaño del fragmento.")

    resultado = []
    for pagina in paginas:
        palabras = pagina["texto"].split()
        inicio = 0
        while inicio < len(palabras):
            fin = min(inicio + tamano_palabras, len(palabras))
            texto = " ".join(palabras[inicio:fin]).strip()
            if texto:
                resultado.append({
                    "archivo": pagina["archivo"],
                    "pagina": pagina["pagina"],
                    "texto": texto
                })
            if fin >= len(palabras):
                break
            inicio = fin - solapamiento
    return resultado

TAMANO_FRAGMENTO = 220
SOLAPAMIENTO = 40
fragmentos = crear_fragmentos(paginas, TAMANO_FRAGMENTO, SOLAPAMIENTO)

if not fragmentos:
    raise ValueError("No se han creado fragmentos a partir de los documentos.")

print("Número de fragmentos:", len(fragmentos))
print("\nEjemplo del primer fragmento:\n")
print(fragmentos[0]["texto"][:700])

Número de fragmentos: 383

Ejemplo del primer fragmento:

O P E N U A X . C O M Módulo 1 La IA en el mundo empresarial L a I A e n e l m u n d o e m p r e s a r i a l : C i c l o d e v i d a , o r g a n i z a c i ó n y r e g u l a c i ó n


## 5. Preparar la búsqueda

En esta parte se prepara la búsqueda semántica. El modelo convierte cada fragmento y cada pregunta en una representación numérica (embedding). Después se calcula la similitud del coseno para ordenar los fragmentos según su relación con la pregunta. Una puntuación alta no garantiza que el fragmento contenga la respuesta correcta.

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Modelo que permite trabajar con textos en español.
MODELO_EMBEDDINGS = "paraphrase-multilingual-MiniLM-L12-v2"

# Cargar el modelo de búsqueda semántica.
modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

# Recoger los textos de los fragmentos creados anteriormente.
textos_fragmentos = [f["texto"] for f in fragmentos]

# Convertir los fragmentos en vectores numéricos.
matriz_embeddings = modelo_embeddings.encode(
    textos_fragmentos,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Modelo:", MODELO_EMBEDDINGS)
print("Fragmentos indexados:", len(textos_fragmentos))
print("Dimensiones de los embeddings:", matriz_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Modelo: paraphrase-multilingual-MiniLM-L12-v2
Fragmentos indexados: 383
Dimensiones de los embeddings: (383, 384)


In [6]:
def recuperar_fragmentos(pregunta, k=4):
    pregunta = str(pregunta).strip()

    if not pregunta:
        raise ValueError("La pregunta no puede estar vacía.")

    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("k debe ser un entero mayor o igual que 1.")

    if not fragmentos:
        return []

    # Convertir la pregunta en un vector numérico.
    vector_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Comparar la pregunta con los fragmentos del PDF.
    puntuaciones = cosine_similarity(
        vector_pregunta, matriz_embeddings
    )[0]

    # Seleccionar los fragmentos con mayor similitud.
    indices = np.argsort(puntuaciones)[::-1][:min(k, len(fragmentos))]

    recuperados = []
    for indice in indices:
        elemento = fragmentos[int(indice)].copy()
        elemento["similitud"] = float(puntuaciones[int(indice)])
        recuperados.append(elemento)

    return recuperados


# Probar la búsqueda sin generar todavía una respuesta con la API.
pregunta_demo = "¿Qué papel tiene la inteligencia artificial en el mundo empresarial?"

for posicion, elemento in enumerate(
    recuperar_fragmentos(pregunta_demo), start=1
):
    print(
        f"\n[{posicion}] {elemento['archivo']} — "
        f"página {elemento['pagina']} — "
        f"similitud {elemento['similitud']:.3f}"
    )
    print(elemento["texto"][:450], "...")



[1] Unidad 1 - La IA en el mundo empresarial_cgranados.pdf — página 41 — similitud 0.890
Tema 2 41 La inteligencia artificial en la organización de las empresas ...

[2] Unidad 1 - La IA en el mundo empresarial_cgranados.pdf — página 5 — similitud 0.808
Introducción – Algunas cifras 5 Según McKinsey, el 56% de las empresas han adoptado inteligencia artificial (IA) en al menos una función de negocio, un dato que se ha mantenido estable desde 2022 tras un crecimiento acelerado en años anteriores. Las organizaciones que implementan IA han reportado incrementos significativos en eficiencia, con mejoras de hasta un 30% en productividad y reducción de costos operativos ...

[3] Unidad 1 - La IA en el mundo empresarial_cgranados.pdf — página 44 — similitud 0.803
 Contextualización de la Inteligencia Artificial en el ámbito empresarial • Orígenes de la IA : Desde la primera regresión logística para modelado de probabilidades hasta la concepción de IBM Watson y de ahí al desarrollo de modelos

## 6. Introducir la clave de OpenAI



In [7]:
import getpass
from openai import OpenAI

CLAVE_API = getpass.getpass("Introduce tu clave API de OpenAI (entrada oculta): ").strip()
if not CLAVE_API:
    raise ValueError("No se ha introducido una clave API.")

cliente = OpenAI(api_key=CLAVE_API)
MODELO_GENERATIVO = "gpt-4o-mini"

# Estos valores son provisionales. Hay que evaluarlos con preguntas reales antes de sacar conclusiones.
TOP_K = 4
UMBRAL_SIMILITUD = 0.20

print("Cliente configurado. Modelo generativo:", MODELO_GENERATIVO)
print("Aviso: el umbral de similitud es un valor inicial, no un resultado validado.")

Introduce tu clave API de OpenAI (entrada oculta): ··········
Cliente configurado. Modelo generativo: gpt-4o-mini
Aviso: el umbral de similitud es un valor inicial, no un resultado validado.


## 7. Crear las respuestas

El chatbot envía al modelo los fragmentos recuperados junto con la pregunta. Se le pide que utilice solo esa información y que reconozca cuándo no es suficiente. Antes de llamar a la API, también se aplica un umbral de similitud. Este filtro es provisional: debe revisarse con las pruebas y no asegura por sí solo que la respuesta sea correcta.

In [8]:
def responder(pregunta, k=TOP_K):
    """
    Busca información en los documentos PDF y devuelve
    los fragmentos más relacionados con la pregunta.

    Esta versión no utiliza la API de OpenAI.
    """

    pregunta = str(pregunta).strip()

    if not pregunta:
        raise ValueError("Escribe una pregunta antes de continuar.")

    # Recuperamos los fragmentos más relacionados con la pregunta.
    fuentes = recuperar_fragmentos(pregunta, k=k)

    # Comprobamos si hay información disponible.
    if not fuentes:
        return {
            "respuesta": (
                "No se ha encontrado texto utilizable "
                "en los documentos cargados."
            ),
            "fuentes": [],
            "fragmentos": []
        }

    # Si la similitud es demasiado baja, no mostramos
    # los fragmentos como si fueran una respuesta fiable.
    if fuentes[0]["similitud"] < UMBRAL_SIMILITUD:
        return {
            "respuesta": (
                "No se ha encontrado información suficientemente "
                "relacionada en el material docente."
            ),
            "fuentes": fuentes,
            "fragmentos": fuentes
        }

    # Construimos una respuesta a partir de los textos recuperados.
    partes_respuesta = []

    for i, fuente in enumerate(fuentes, start=1):
        partes_respuesta.append(
            f"[{i}] Documento: {fuente['archivo']}\n"
            f"Página: {fuente['pagina']}\n"
            f"Similitud: {fuente['similitud']:.3f}\n\n"
            f"{fuente['texto']}"
        )

    texto_respuesta = (
        "Estos son los fragmentos del material docente "
        "más relacionados con tu pregunta:\n\n"
        + "\n\n---\n\n".join(partes_respuesta)
    )

    return {
        "respuesta": texto_respuesta,
        "fuentes": fuentes,
        "fragmentos": fuentes
    }


def consultar(pregunta):
    """
    Muestra los fragmentos recuperados y sus referencias.
    """

    resultado = responder(pregunta)

    print("RESULTADO DE LA CONSULTA\n")
    print(resultado["respuesta"])

    return resultado



In [9]:
def consultar_por_palabras_clave(pregunta, k=8):
    """
    Busca fragmentos mediante similitud semántica y,
    cuando detecta palabras importantes, prioriza
    los textos que las contienen.
    """

    # Recuperamos más fragmentos que en la búsqueda habitual.
    fuentes = recuperar_fragmentos(pregunta, k=20)

    pregunta_min = pregunta.lower()

    # Palabras clave para este ejemplo del material docente.
    palabras_clave = []

    if "amazon" in pregunta_min or "recomendación" in pregunta_min:
        palabras_clave = ["amazon", "recomendación", "recomendaciones"]

    elif "tesla" in pregunta_min:
        palabras_clave = ["tesla"]

    if palabras_clave:
        fuentes.sort(
            key=lambda fuente: (
                sum(
                    palabra in fuente["texto"].lower()
                    for palabra in palabras_clave
                ),
                fuente["similitud"]
            ),
            reverse=True
        )

    # Mostramos los primeros resultados tras ordenarlos.
    fuentes = fuentes[:k]

    if not fuentes:
        print("No se han encontrado fragmentos.")
        return []

    for i, fuente in enumerate(fuentes, start=1):
        print(
            f"\n[{i}] {fuente['archivo']} | "
            f"Página {fuente['pagina']} | "
            f"Similitud {fuente['similitud']:.3f}"
        )
        print(fuente["texto"])
        print("-" * 60)

    return fuentes


In [10]:

import re

def consultar_resumida(pregunta, k=8):
    """
    Recupera información de los PDF y presenta las frases
    más relevantes de forma ordenada, con sus fuentes.
    No utiliza la API de OpenAI.
    """

    # Buscamos más fragmentos para aumentar las posibilidades
    # de encontrar la información adecuada.
    fuentes = recuperar_fragmentos(pregunta, k=20)

    if not fuentes:
        print("No se ha encontrado información en los documentos.")
        return []

    # Priorizamos los fragmentos con mayor similitud.
    fuentes = sorted(
        fuentes,
        key=lambda fuente: fuente["similitud"],
        reverse=True
    )

    # Conservamos los fragmentos cercanos a la mejor puntuación.
    mejor_similitud = fuentes[0]["similitud"]

    fuentes_relevantes = [
        fuente for fuente in fuentes
        if fuente["similitud"] >= max(
            UMBRAL_SIMILITUD,
            mejor_similitud - 0.18
        )
    ][:k]

    print("RESPUESTA BASADA EN EL MATERIAL DOCENTE\n")

    frases_mostradas = 0
    frases_vistas = set()

    for fuente in fuentes_relevantes:
        # Separamos el texto recuperado en frases.
        frases = re.split(
            r"(?<=[.!?])\s+",
            fuente["texto"].replace("\n", " ")
        )

        frases_validas = []

        for frase in frases:
            frase = frase.strip()

            # Ignoramos frases demasiado cortas o repetidas.
            if len(frase) < 35:
                continue

            if frase.lower() in frases_vistas:
                continue

            frases_validas.append(frase)
            frases_vistas.add(frase.lower())

        if not frases_validas:
            continue

        print(
            f"Fuente: {fuente['archivo']} "
            f"(página {fuente['pagina']})"
        )

        for frase in frases_validas:
            print(f"• {frase}")
            frases_mostradas += 1

        print()

    if frases_mostradas == 0:
        print("No se han encontrado frases que mostrar.")

    print(
        "Nota: el sistema organiza fragmentos recuperados; "
        "no genera un resumen mediante un modelo de lenguaje."
    )

    return fuentes_relevantes


## 8. Hacer una pregunta

Prueba con una pregunta que se pueda responder con el PDF. La celda muestra la respuesta y los fragmentos que ha usado, para que puedas revisarlos.

In [38]:
pregunta = input(
    "Escribe tu pregunta sobre el material docente: "
).strip()

if pregunta:
    ultimo_resultado = consultar_resumida(pregunta)
else:
    print("No se ha introducido ninguna pregunta.")



Escribe tu pregunta sobre el material docente: ¿Qué conclusión presenta el material sobre un estudio realizado en 2026 que compara todos los modelos de inteligencia artificial empresarial?
RESPUESTA BASADA EN EL MATERIAL DOCENTE

Fuente: Unidad 1 - La IA en el mundo empresarial_cgranados.pdf (página 5)
• Introducción – Algunas cifras 5 Según McKinsey, el 56% de las empresas han adoptado inteligencia artificial (IA) en al menos una función de negocio, un dato que se ha mantenido estable desde 2022 tras un crecimiento acelerado en años anteriores.
• Las organizaciones que implementan IA han reportado incrementos significativos en eficiencia, con mejoras de hasta un 30% en productividad y reducción de costos operativos

Fuente: Unidad 1 - La IA en el mundo empresarial_cgranados.pdf (página 9)
• Introducción - Pregunta 9 ¿En qué sectores creéis que la IA tendrá mayor impacto en los próximos 5 años?

Fuente: Unidad 1 - La IA en el mundo empresarial_cgranados.pdf (página 41)
• Tema 2 41 La i

## 9. Guardar las pruebas

En la evaluación conviene distinguir entre la calidad de la búsqueda y la calidad de la respuesta. Por eso se guardan la pregunta, los fragmentos recuperados y la respuesta. Las puntuaciones se dejan vacías para completarlas después de comparar el resultado con el PDF.

In [12]:
from datetime import datetime
import pandas as pd

TIPOS_PREGUNTA = ["directa", "reformulada", "multifragmento", "ambigua", "sin_evidencia"]
registros = []

def registrar_prueba(pregunta, tipo_pregunta, respuesta_esperada="", observaciones=""):
    # Ejecuta una consulta y guarda sus datos para evaluarlos más tarde.
    if not str(pregunta).strip():
        raise ValueError("La pregunta no puede estar vacía.")
    if tipo_pregunta not in TIPOS_PREGUNTA:
        raise ValueError("tipo_pregunta debe ser uno de: " + ", ".join(TIPOS_PREGUNTA))

    resultado = responder(pregunta)
    registros.append({
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "tipo_pregunta": tipo_pregunta,
        "pregunta": pregunta,
        "respuesta_esperada_y_fuente": respuesta_esperada,
        "fragmentos_recuperados": "\n\n".join(
            f"{f['archivo']} (p. {f['pagina']}; similitud={f['similitud']:.3f}): {f['texto']}"
            for f in resultado["fragmentos"]
        ),
        "respuesta_chatbot": resultado["respuesta"],
        "correccion_1a5": "",
        "relevancia_1a5": "",
        "fidelidad_contexto_1a5": "",
        "claridad_1a5": "",
        "tipo_error": "",
        "observaciones": observaciones
    })
    print(resultado["respuesta"])
    return resultado

def ver_registro():
    columnas = [
        "fecha", "tipo_pregunta", "pregunta", "respuesta_esperada_y_fuente",
        "fragmentos_recuperados", "respuesta_chatbot", "correccion_1a5",
        "relevancia_1a5", "fidelidad_contexto_1a5", "claridad_1a5",
        "tipo_error", "observaciones"
    ]
    return pd.DataFrame(registros, columns=columnas)

def exportar_registro(nombre="registro_pruebas_chatbot_tfm.csv"):
    if not registros:
        print("Todavía no hay pruebas registradas. Ejecuta registrar_prueba(...) primero.")
        return
    # utf-8-sig ayuda a que Excel interprete correctamente los acentos.
    ver_registro().to_csv(nombre, index=False, encoding="utf-8-sig")
    print("Registro exportado:", nombre)
    files.download(nombre)

print("Tipos de pregunta disponibles:", ", ".join(TIPOS_PREGUNTA))

Tipos de pregunta disponibles: directa, reformulada, multifragmento, ambigua, sin_evidencia


## 10. Descargar el registro

Cuando se hayan añadido varias pruebas, se ejecuta la celda siguiente para descargar un archivo CSV.

In [14]:
exportar_registro()

Todavía no hay pruebas registradas. Ejecuta registrar_prueba(...) primero.
